# 🥇 Notebook 04: Gold Analytical Layer & Delta Lake Physical Optimization (`OPTIMIZE` & `Z-ORDER BY`)

## 🎯 Objectives & "Why We Do This" Comparative Proofs
1. **Gold Analytical Layer**: Create business aggregations (`gold_daily_sales` & `gold_customer_360`).
2. **Small File Problem & `OPTIMIZE` Proof**: Comparative metadata metrics showing file count reduction from 26 files to 1 file.
3. **Multi-Dimensional Clustering & `Z-ORDER BY` Proof**: Comparative query execution speed benchmarks demonstrating **Delta Data Skipping**.

---

In [0]:
# Databricks notebook source
import time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

SILVER_PATH = "/tmp/mini_project2/silver"
GOLD_PATH = "/tmp/mini_project2/gold"

print(f"Reading Silver Tables from: {SILVER_PATH}")
print(f"Writing Gold Delta Tables to: {GOLD_PATH}")

### 💡 STEP 1: Build Gold Layer Business Aggregations

In [0]:
df_orders = spark.read.format("delta").load(f"{SILVER_PATH}/orders")
df_cust = spark.read.format("delta").load(f"{SILVER_PATH}/customers")

# 1. Daily Sales Summary
df_gold_daily = df_orders.join(df_cust, "customer_id") \
    .groupBy("order_date", "region", "tier") \
    .agg(
        F.count("order_id").alias("total_orders"),
        F.sum("total_amount").alias("total_revenue"),
        F.avg("total_amount").alias("avg_order_value")
    ) \
    .withColumn("_gold_updated_at", F.current_timestamp())

df_gold_daily.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/daily_sales")
spark.sql(f"CREATE TABLE IF NOT EXISTS gold_daily_sales USING DELTA LOCATION '{GOLD_PATH}/daily_sales'")

# 2. Customer 360 Profile
df_customer_360 = df_orders.join(df_cust, "customer_id") \
    .groupBy("customer_id", "customer_name", "region", "tier") \
    .agg(
        F.count("order_id").alias("lifetime_orders"),
        F.sum("total_amount").alias("lifetime_spend"),
        F.max("order_date").alias("last_order_date")
    ) \
    .withColumn("customer_spend_tier", 
        F.when(F.col("lifetime_spend") > 2000, "VIP")
         .when(F.col("lifetime_spend") > 1000, "High_Value")
         .otherwise("Standard")
    )

df_customer_360.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/customer_360")
spark.sql(f"CREATE TABLE IF NOT EXISTS gold_customer_360 USING DELTA LOCATION '{GOLD_PATH}/customer_360'")

print("=== 📊 DEMONSTRATION: Gold Layer Aggregations ===")
print(f"✅ gold_daily_sales created: {df_gold_daily.count()} summary rows")
print(f"✅ gold_customer_360 created: {df_customer_360.count()} profiles")
print("💡 WHY WE BUILD GOLD LAYER: Gold tables pre-aggregate complex joins and metrics into business-ready star-schema / dimension tables so BI dashboards (PowerBI, Tableau) load in milliseconds without re-computing expensive joins.")

--- 
## 🧪 PROOF 1: The Small File Problem & `OPTIMIZE` File Compaction

### Why do we need `OPTIMIZE`?
When streaming or ingesting micro-batches, Delta Lake writes thousands of tiny Parquet files. Reading tiny files introduces huge filesystem metadata overhead.

In [0]:
# Simulate Small File Problem by appending 25 small micro-batches
print("=== SIMULATING SMALL FILE FRAGMENTATION ===")
sample_chunk = df_gold_daily.limit(50)
for i in range(25):
    sample_chunk.write.format("delta").mode("append").save(f"{GOLD_PATH}/daily_sales")

print("✅ Injected 25 small append transactions into gold_daily_sales.")

In [0]:
%sql
-- Inspect Table Detail BEFORE OPTIMIZE (Check numFiles count)
DESCRIBE DETAIL gold_daily_sales;

In [0]:
%sql
-- 🛠️ RUN OPTIMIZE to Compact Small Files
OPTIMIZE gold_daily_sales;

In [0]:
%sql
-- Inspect Table Detail AFTER OPTIMIZE (Notice numFiles reduced to 1 file!)
DESCRIBE DETAIL gold_daily_sales;

In [0]:
print("=== 📊 DEMONSTRATION VERDICT: OPTIMIZE (File Compaction) ===")
print("💡 WHY WE USE OPTIMIZE: OPTIMIZE merges 25+ tiny fragmented files into 1 optimal Parquet file, reducing filesystem I/O operations by ~96% and dramatically accelerating query read speeds!")

--- 
## 🧪 PROOF 2: `Z-ORDER BY` Multi-Dimensional Clustering & Data Skipping

### Why do we need `Z-ORDER BY`?
When filtering non-partitioned tables by unique keys (e.g. `customer_id`), standard tables must scan *every single Parquet file*. Z-Ordering clusters data by `customer_id` inside Parquet files, allowing Delta Lake to **skip reading unneeded files entirely**!

In [0]:
# Benchmark Query BEFORE Z-Ordering
target_customer = "CUST_001234"
print(f"=== 📊 BENCHMARK QUERY BEFORE Z-ORDER (Filtering customer_id = '{target_customer}') ===")

t0 = time.time()
res_before = spark.sql(f"SELECT * FROM gold_customer_360 WHERE customer_id = '{target_customer}'").collect()
t1 = time.time()
duration_before = round(t1 - t0, 4)
print(f"⏱️ Query Duration BEFORE Z-Order: {duration_before} seconds")

In [0]:
%sql
-- 🛠️ EXECUTE OPTIMIZE WITH ZORDER BY (customer_id, region)
OPTIMIZE gold_customer_360 
ZORDER BY (customer_id, region);

In [0]:
# Benchmark Query AFTER Z-Ordering
print(f"=== 📊 BENCHMARK QUERY AFTER Z-ORDER (Filtering customer_id = '{target_customer}') ===")

t0 = time.time()
res_after = spark.sql(f"SELECT * FROM gold_customer_360 WHERE customer_id = '{target_customer}'").collect()
t1 = time.time()
duration_after = round(t1 - t0, 4)
print(f"⏱️ Query Duration AFTER Z-Order: {duration_after} seconds")

if duration_before > 0:
    speedup = round((duration_before - duration_after) / duration_before * 100, 2)
    print(f"🚀 Performance Improvement: {speedup}% faster!")
print("💡 WHY WE USE Z-ORDER BY: Z-Ordering co-locates rows by customer_id within Parquet files. When filtering by customer_id, Delta reads min/max file statistics in _delta_log and skips scanning non-matching Parquet files!")